# PM4Py

In [ ]:
import pm4py

In [ ]:
path = "data/scenario_test_ocel.json"  # example_3_ocel
ocel = pm4py.read_ocel2_json(path)

In [ ]:
df_object_type_activities = pm4py.ocel_object_type_activities(ocel)
df_ocel_temporal_summary = pm4py.ocel_temporal_summary(ocel)
df_objects_summary = pm4py.ocel_objects_summary(ocel)

In [ ]:
ocdfg = pm4py.discover_ocdfg(ocel)

# View the model with the frequency annotation
pm4py.view_ocdfg(ocdfg, format="png")

In [ ]:
from pm4py.algo.transformation.ocel.graphs import object_interaction_graph

graph = object_interaction_graph.apply(ocel)

In [ ]:
model = pm4py.discover_oc_petri_net(ocel)
ocpn_view = pm4py.view_ocpn(model, format="png")

In [ ]:
df_events = ocel.events.copy()
df_events.set_index("ocel:eid", inplace=True)
df_relations = ocel.relations.copy()
df_relations.set_index("ocel:eid", inplace=True)

df_events_objects = df_events.join(df_relations, rsuffix="_relations")

In [ ]:
df_objects = ocel.objects.copy()
df_objects.set_index("ocel:oid", inplace=True)

In [ ]:
df_events.groupby("ocel:activity").describe()

# Networkx

In [ ]:
import networkx as nx

In [ ]:
_ocel_nx = pm4py.convert.convert_ocel_to_networkx(ocel)

# Workaround for https://github.com/process-intelligence-solutions/pm4py/issues/534
ocel_nx = nx.MultiDiGraph()
ocel_nx.add_nodes_from(_ocel_nx.nodes(data=True))
ocel_nx.add_edges_from(
    [e for e in _ocel_nx.edges(data=True) if e[-1]["attr"].get("type") != "DF"]
)

from typing import List


def lifecycle_max_lower_than(lif: List[str], e_prime: str):
    lif_int = [int(e) for e in lif]
    return str(max([e for e in lif_int if e < int(e_prime)]))


selected_aggregation_activity_qualifier = [
    ("Aggregation-ADD", "childObject"),
    # ("Aggregation-DELETE", "parentObject"),
]
agg_act_qual = [
    f"{act}-{qual}" for act, qual in selected_aggregation_activity_qualifier
]
ocel.relations["activity-qualifier"] = (
    ocel.relations["ocel:activity"] + "-" + ocel.relations["ocel:qualifier"]
)

lifecycle = (
    ocel.relations.groupby(ocel.object_id_column)
    .agg(list)
    .to_dict()[ocel.event_id_column]
)
for obj in lifecycle:
    # Add DF edges
    lif = lifecycle[obj]
    for i in range(len(lif) - 1):
        ocel_nx.add_edge(lif[i], lif[i + 1], attr={"type": "DF", "object": obj})

    # Add aggregation DF edges
    for activity, qualifier in selected_aggregation_activity_qualifier:
        relations_obj = ocel.relations[ocel.relations[ocel.object_id_column] == obj]
        lif = lifecycle[obj]
        # Add DF_agg for selected aggregation events, taking into account the E2O qualifier
        for event in relations_obj[
            (relations_obj["ocel:activity"] == activity)
            & (relations_obj["ocel:qualifier"] == qualifier)
        ][ocel.event_id_column].values:
            # Get preceding event from events that are not in the selected aggregation activity - qualifier pairs
            lif_selected = relations_obj[
                ~relations_obj["activity-qualifier"].isin(agg_act_qual)
            ][ocel.event_id_column].values
            ocel_nx.add_edge(
                lifecycle_max_lower_than(lif_selected, event),
                event,
                attr={"type": "DF_agg", "object": obj},
            )

In [ ]:
# Store graph to GraphML format

import json

from pandas import Timestamp

def convert_timestamp_to_iso():
    for n, d in ocel_nx.nodes(data=True):
        if not d.get("attr", {}).get("ocel:timestamp"):
            continue
        if isinstance(d["attr"]["ocel:timestamp"], Timestamp):
            d["attr"]["ocel:timestamp"] = d["attr"]["ocel:timestamp"].isoformat()

# Convert node attributes that are dictionaries into JSON strings so GraphML can store them
convert_timestamp_to_iso()

for n, d in ocel_nx.nodes(data=True):
    for k, v in list(d.items()):
        if isinstance(v, dict):
            try:
                d[k] = json.dumps(v)
            except Exception:
                d[k] = str(v)

# Convert edge attributes (handle MultiDiGraph and DiGraph)
try:
    # MultiGraph/MultiDiGraph edges include keys
    for u, v, key, ed in ocel_nx.edges(keys=True, data=True):
        for k, val in list(ed.items()):
            if isinstance(val, dict):
                try:
                    ed[k] = json.dumps(val)
                except Exception:
                    ed[k] = str(val)
except TypeError:
    # fallback for Graph/DiGraph without keys
    for u, v, ed in ocel_nx.edges(data=True):
        for k, val in list(ed.items()):
            if isinstance(val, dict):
                try:
                    ed[k] = json.dumps(val)
                except Exception:
                    ed[k] = str(val)

nx.write_graphml(ocel_nx, path=path.replace(".json", ".graphml"))

In [ ]:
# Load the graphml and parse JSON attributes back
def load_graphml_with_json_attrs(path: str) -> nx.Graph:
    """Read a GraphML file and attempt to JSON-decode any string attributes back into Python objects.

    Only replaces attribute values when json.loads returns a dict or list (to avoid converting plain strings).
    Works for Graph/DiGraph and MultiGraph/MultiDiGraph edge representations.
    """
    G = nx.read_graphml(path)

    # Nodes
    for n, d in G.nodes(data=True):
        for k, v in list(d.items()):
            if isinstance(v, str):
                try:
                    parsed = json.loads(v)
                    if isinstance(parsed, (dict, list)):
                        d[k] = parsed
                except Exception:
                    # leave as string if it isn't JSON
                    pass

    # Edges (handle keyed MultiGraphs and non-keyed graphs)
    try:
        edges = list(G.edges(keys=True, data=True))
        keyed = True
    except TypeError:
        edges = list(G.edges(data=True))
        keyed = False

    if keyed:
        for u, v, key, ed in edges:
            for k, val in list(ed.items()):
                if isinstance(val, str):
                    try:
                        parsed = json.loads(val)
                        if isinstance(parsed, (dict, list)):
                            ed[k] = parsed
                    except Exception:
                        pass
    else:
        for u, v, ed in edges:
            for k, val in list(ed.items()):
                if isinstance(val, str):
                    try:
                        parsed = json.loads(val)
                        if isinstance(parsed, (dict, list)):
                            ed[k] = parsed
                    except Exception:
                        pass

    return G

graphml_path = path.replace(".json", ".graphml")
ocel_nx = load_graphml_with_json_attrs(graphml_path)

In [ ]:
object_types = ["PackingUnit"]

# object_types = ["HingePack"]
# activities = ["PackHinges"]

events_to_trace = df_events_objects[
    (df_events_objects["ocel:type"].isin(object_types))
].index.values

print(f"Number of events selected: {len(events_to_trace)}")

In [ ]:
from importlib import reload

import process_execution

reload(process_execution)

In [ ]:
from collections import Counter

from process_execution import extract_process_execution


def determine_class_quality(event: str):
    return ocel_nx.nodes()[event]["attr"].get("averageQuality") >= 1.0


def determine_class_attribute(trace_graph: nx.Graph):
    selected_activity = "Object-departing-WB"
    selected_attribute = "a"
    for _, data in trace_graph.nodes(data="attr"):
        if (
            data.get("ocel:activity", "") == selected_activity
            and data.get(selected_attribute, 1) < 0.25
        ):
            return False
    return True


trace_graphs = {}
for event in events_to_trace:
    trace_graph = extract_process_execution(
        ocel_nx,
        event,
        ["ProductionLot", "PackingUnit"],
        "Object-creating_class_instance",
    )
    trace_graph.construct_node_label()
    trace_graph.construct_edge_label()

    trace_graphs[event] = {
        "process_execution": trace_graph,
        # "class": determine_class_quality(event),
        "class": determine_class_attribute(trace_graph),
    }


Counter([d["class"] for d in trace_graphs.values()])

In [ ]:
for target_event, trace_graph in trace_graphs.items():
    normalize_events = [
        node
        for node, attr in trace_graph["process_execution"].nodes(data=True)
        if attr["attr"].get("ocel:activity", "") == "Aggregation-ADD"
    ]

    trace_graphs[target_event]["normalized_process_executions"] = [
        p
        for p in trace_graph["process_execution"].extract_normalized_process_executions(
            target_event, normalize_events
        )
    ]

In [ ]:
from grakel.kernels import (
    RandomWalk,
    SubgraphMatching,
    VertexHistogram,
    WeisfeilerLehman,
)
from grakel.utils import graph_from_networkx

# Construct node label from attributes
for trace_dict in trace_graphs.values():
    trace_graph = trace_dict["graph"]
    construct_node_label(trace_graph)
    # select_node_attr(trace_graph, attr_key="s_co2e[kg]")

# Select graphs and convert to DiGraph
selected_trace_graphs = {k: v["graph"] for k, v in list(trace_graphs.items())[:100]}

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="label",
    # as_Graph=True,
    # val_node_labels="test",
    # edge_labels_tag="attr",
)
gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)
r_gk = gk.fit_transform(selected_trace_graphs_grakel)


def numeric_diff(a, b):
    try:
        return 1 - (float(b) - float(a)) / float(a)
    except ZeroDivisionError:
        return 0.5
    except ValueError:
        return 0.5


def dict_compare(a, b):
    sim_score = 0
    for key in a.keys():
        v_a = a.get(key)
        v_b = b.get(key)

        if not (v_a and v_b):
            sim_score += 0

        try:
            sim_score += 1 - (float(v_b) - float(v_a)) / float(v_a)
        except ZeroDivisionError:
            sim_score += 0.5
        except TypeError:
            sim_score += int(v_a == v_b)
        except ValueError:
            sim_score += int(v_a == v_b)
    return sim_score


# sub_match = SubgraphMatching(
#     normalize=True,
#     kv=dict_compare,
#     ke=None,
# )
# r_sub_match = sub_match.fit_transform(trace_graphs_grakel)

# trace_graphs_grakel = graph_from_networkx(
#     trace_graphs.values(),
#     node_labels_tag="label",
#     as_Graph=True,
# )
# random_walk = RandomWalk(n_jobs=1, normalize=True)
# r_random_walk = random_walk.fit_transform(trace_graphs_grakel)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# pcolormesh (matplotlib) — very fast for large arrays, no text annotation
plt.figure(figsize=(12, 10))
plt.pcolormesh(df_r.values, cmap="coolwarm")
plt.colorbar(label="Normalized similarity")
plt.gca().set_xticks(np.arange(df_r.shape[1]) + 0.5)
plt.gca().set_yticks(np.arange(df_r.shape[0]) + 0.5)
plt.gca().set_xticklabels(df_r.columns, rotation=90, fontsize=8)
plt.gca().set_yticklabels(df_r.index, fontsize=8)
plt.title("Weisfeiler-Lehman subtree kernel")
plt.tight_layout()
plt.savefig("figures/plot.png")  # Save the figure

### Instance-based counterfactual

Two step approach:
1) Find *k* graphs with different class, but similar structure (including node labels);
2) Among the *k* graphs, find the most similar graph, also considering the node attributes.

In [ ]:
from grakel.kernels import (
    VertexHistogram,
    WeisfeilerLehman,
)
from grakel.utils import graph_from_networkx

import numpy as np

k = 10  # select top k structurally most similar graphs

target_trace_graph_id = "100023"
target_trace_graph = trace_graphs[target_trace_graph_id]["process_execution"]

selected_trace_graphs = {
    k: trace_graphs[k]["process_execution"] for k in (list(trace_graphs.keys()))
}

gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
gk.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
K_gk = gk.transform(selected_trace_graphs_grakel)

most_similar_trace_graphs_gk = np.array(list(selected_trace_graphs.keys()))[
    np.argsort(K_gk[:, 0])[(-1 * k) :]
]
print(most_similar_trace_graphs_gk)

In [ ]:
import numpy as np

from grakel.kernels import SubgraphMatching

selected_trace_graphs = {
    k: nx.DiGraph(trace_graphs[k]["process_execution"])
    for k in most_similar_trace_graphs_gk
    if trace_graphs[k]["class"] != trace_graphs[target_trace_graph_id]["class"]
}


def numeric_diff(a, b):
    try:
        return 1 - (float(b) - float(a)) / float(a)
    except ZeroDivisionError:
        return 0.5
    except ValueError:
        return 0.5


def dict_compare(a, b):
    sim_score = 0
    for key in a.keys():
        v_a = a.get(key)
        v_b = b.get(key)

        if not (v_a and v_b):
            sim_score += 0

        try:
            sim_score += 1 - (float(v_b) - float(v_a)) / float(v_a)
        except ZeroDivisionError:
            sim_score += 0.5
        except TypeError:
            sim_score += int(v_a == v_b)
        except ValueError:
            sim_score += int(v_a == v_b)
    return sim_score


sub_match = SubgraphMatching(
    normalize=True,
    kv=dict_compare,
    ke=None,
)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    # edge_labels_tag="attr",
)
sub_match.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    edge_labels_tag="attr",
)
K = sub_match.transform(selected_trace_graphs_grakel)

most_similar_trace_graph_id = list(selected_trace_graphs.keys())[
    np.argsort(K[:, 0])[-1]
]

print("Query process execution")
print("--------------")
print(target_trace_graph_id)
print()
print("Most similar process execution")
print("---------------------")
print(most_similar_trace_graph_id)

### Visualization

In [ ]:
from importlib import reload
import visualization

reload(visualization)

In [ ]:
from analysis.visualization import (
    apply_node_styles_nx,
    apply_edge_styles_nx,
    visualize_with_svg_and_js,
    visualize_highlight_normalized,
)

selected_for_visualization = [target_trace_graph_id, most_similar_trace_graph_id]
for event, event_dict in trace_graphs.items():
    if event not in selected_for_visualization:
        continue

    trace_graph = event_dict["process_execution"]

    apply_node_styles_nx(trace_graph)  # apply coloring + tooltip
    apply_edge_styles_nx(trace_graph)  # apply coloring + tooltip

    # Draw base process execution graph
    agraph = nx.nx_agraph.to_agraph(trace_graph)
    agraph.draw(f"figures/{event}.svg", prog="dot")

    # Visualize normalized subgraphs (if present)
    normalized = event_dict.get("normalized_process_executions", [])

    try:
        html_path_svg = visualize_with_svg_and_js(
            trace_graph, normalized, out_prefix=event
        )
        print(f"Wrote agraph-layout interactive visualization: {html_path_svg}")
    except Exception as e:
        print(f"Failed to create SVG+JS visualization for {event}: {e}")

    # If normalized subgraphs exist, render each with highlighted nodes/edges
    try:
        out_paths_svg = visualize_highlight_normalized(
            trace_graph, normalized, out_prefix=event
        )
        print(
            f"Wrote visualization(s) with highlighted normalized process execution: {out_paths_svg}"
        )
    except Exception as e:
        print(f"Failed to create SVG visualization for {event}: {e}")


# apply_node_styles_nx(ocel_nx)  # apply coloring + tooltip
# agraph = nx.nx_agraph.to_agraph(ocel_nx)
# agraph.draw("figures/example_1_ocel.svg", prog="dot")

#### Graph alignment

In [ ]:
import functools
import numpy as np  # numpy backend
import pygmtools as pygm
import matplotlib.pyplot as plt  # for plotting
import networkx as nx  # for plotting graphs
import torch
from matplotlib.patches import ConnectionPatch  # for plotting matching result

# pygm.set_backend("numpy")  # set default backend for pygmtools
pygm.set_backend("pytorch")
np.random.seed(1)  # fix random seed

In [ ]:
graph_1_id = "100023"
graph_2_id = "449315"
attr_exclude_show = ["ocel:eid", "ocel:timestamp"]

G1 = trace_graphs[graph_1_id]["process_execution"]
G2 = trace_graphs[graph_2_id]["process_execution"]

n1 = len(G1.nodes)
n2 = len(G2.nodes)
A1 = torch.from_numpy(nx.to_numpy_array(G1))
A2 = torch.from_numpy(nx.to_numpy_array(G2))

conn1, edge1 = pygm.utils.dense_to_sparse(A1)
conn2, edge2 = pygm.utils.dense_to_sparse(A2)

gaussian_aff = functools.partial(
    pygm.utils.gaussian_aff_fn, sigma=0.1
)  # set affinity function
K = pygm.utils.build_aff_mat(
    None,
    edge1,
    conn1,
    None,
    edge2,
    conn2,
    None,
    None,
    None,
    None,
    edge_aff_fn=gaussian_aff,
)

# X = pygm.sm(K, n1, n2)
X = pygm.rrwm(K, n1, n2)
X = pygm.hungarian(X)

for i, g_i in enumerate(G1.nodes):
    g_j = list(G2.nodes)[np.argmax(X[i]).item()]
    a_i = G1.nodes.data()[g_i]["attr"]
    a_j = G2.nodes.data()[g_j]["attr"]

    diff = {
        k: (a_i[k], a_j[k])
        for k in a_i
        if k in a_j and a_i[k] != a_j[k] and k not in attr_exclude_show
    }
    print(g_i, g_j, diff)

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder


def label_encoder(G: nx.Graph, node_edge: str):
    if node_edge == "node":
        labels = [n[1]["label"] for n in G.nodes(data=True)]
    elif node_edge == "edge":
        labels = [n[-1]["label"] for n in G.edges(data=True, keys=True)]
    else:
        raise ValueError(f"node_edge={node_edge} not valid")

    # Integer encoding
    label_encoder = LabelEncoder()
    int_labels = label_encoder.fit_transform(labels)
    # print("Integer encoded labels:", int_labels)

    # One-hot encoding (optional, for ML models)
    onehot_encoder = OneHotEncoder(sparse_output=False)
    int_labels_reshaped = int_labels.reshape(-1, 1)
    onehot_labels = onehot_encoder.fit_transform(int_labels_reshaped)
    # print("One-hot encoded labels:\n", onehot_labels)

    # Store encoded labels back into the graph
    if node_edge == "node":
        for idx, node in enumerate(G.nodes()):
            G.nodes[node]["label_id"] = int_labels[idx]
            G.nodes[node]["label_onehot"] = onehot_labels[idx]
    elif node_edge == "edge":
        for idx, edge in enumerate(G.edges(keys=True)):
            G.edges[edge]["label_id"] = int_labels[idx]
            G.edges[edge]["label_onehot"] = onehot_labels[idx]


label_encoder(G1, node_edge="node")
label_encoder(G2, node_edge="node")
label_encoder(G1, node_edge="edge")
label_encoder(G2, node_edge="edge")

In [ ]:
_feat1 = torch.tensor(
    np.array([n[1]["label_onehot"] for n in G1.nodes(data=True)]), dtype=torch.float32
)
_feat2 = torch.tensor(
    np.array([n[1]["label_onehot"] for n in G2.nodes(data=True)]), dtype=torch.float32
)
projector = torch.nn.Linear(_feat1.shape[-1], 1024, dtype=torch.float32)
feat1 = projector(_feat1)
feat2 = projector(_feat2)

# e_feat1 = np.array([[e[2]["attr"]["type"] for e in G1.edges(data=True)]])
# e_feat2 = np.array([[e[2]["attr"]["type"] for e in G2.edges(data=True)]])
e_feat1 = torch.from_numpy(np.array([nx.to_numpy_array(G1)]).T).to(torch.float32)
e_feat2 = torch.from_numpy(np.array([nx.to_numpy_array(G2)]).T).to(torch.float32)

X, net = pygm.cie(
    feat1,
    feat2,
    A1.to(torch.float32),
    A2.to(torch.float32),
    e_feat1,
    e_feat2,
    torch.tensor([n1]),
    torch.tensor([n2]),
    return_network=True,
)
X = pygm.hungarian(X)

In [ ]:
X = pygm.ipca_gm(feat1, feat2, A1.to(torch.float32), A2.to(torch.float32))
X = pygm.hungarian(X)

In [ ]:
# Visualization
pos1 = nx.drawing.nx_pydot.pydot_layout(G1)
pos2 = nx.drawing.nx_pydot.pydot_layout(G2)

plt.figure(figsize=(8, 4))
ax1 = plt.subplot(1, 2, 1)
plt.title(f"Graph 1 - {graph_1_id}")
nx.draw_networkx(G1, pos1)
ax2 = plt.subplot(1, 2, 2)
plt.title(f"Graph 2 - {graph_2_id}")
nx.draw_networkx(G2, pos2)

X_np = X.detach().numpy()
for i, g_i in enumerate(G1.nodes):
    j = np.argmax(X_np[i]).item()
    g_j = list(G2.nodes)[j]
    con = ConnectionPatch(
        xyA=pos1[g_i],
        xyB=pos2[g_j],
        coordsA="data",
        coordsB="data",
        axesA=ax1,
        axesB=ax2,
        color="green",
    )
    plt.gca().add_artist(con)

In [ ]:
pos1 = nx.drawing.nx_pydot.pydot_layout(G1)
pos2 = nx.drawing.nx_pydot.pydot_layout(G2)

X_np = X.detach().numpy()
A2_np = A2.detach().numpy()
align_A2 = np.matmul(np.matmul(X_np, A2_np), X_np.T)
plt.figure(figsize=(8, 4))
ax1 = plt.subplot(1, 2, 1)
plt.title(f"Graph 1 - {graph_1_id}")
nx.draw_networkx(G1, pos=pos1)
ax2 = plt.subplot(1, 2, 2)
plt.title(f"Aligned Graph 2 - {graph_2_id}")

align_pos2 = {}
for i, g_i in enumerate(G1.nodes):
    j = np.argmax(X_np[i]).item()
    g_j = list(G2.nodes)[j]
    align_pos2[g_j] = pos1[g_i]
    con = ConnectionPatch(
        xyA=pos1[g_i],
        xyB=align_pos2[g_j],
        coordsA="data",
        coordsB="data",
        axesA=ax1,
        axesB=ax2,
        color="green",
    )
    plt.gca().add_artist(con)

    a_i = G1.nodes.data()[g_i]["attr"]
    a_j = G2.nodes.data()[g_j]["attr"]
    diff = {
        k: (a_i[k], a_j[k])
        for k in a_i
        if k in a_j and a_i[k] != a_j[k] and k not in attr_exclude_show
    }
    if diff:
        print(g_i, g_j, diff)

unaligned_nodes = set(pos2.keys()) - set(align_pos2.keys())

pos2.update(align_pos2)
nx.draw_networkx(G2, pos=pos2)

print(unaligned_nodes)

In [ ]:
target_trace_graph_id, most_similar_trace_graph_id

#### Heatmaps (similarity)

In [ ]:
import matplotlib.pyplot as plt
from pandas import DataFrame

df_r = DataFrame(
    r_gk, index=selected_trace_graphs.keys(), columns=selected_trace_graphs.keys()
)

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

method = "average"
metric = "correlation"
# compute linkage on rows/cols
Lr = linkage(df_r.values, method=method, metric=metric)
Lc = linkage(df_r.values.T, method=method, metric=metric)
ridx = leaves_list(Lr)
cidx = leaves_list(Lc)
ordered = df_r.iloc[ridx, :].iloc[:, cidx]
plt.figure(figsize=(12, 10))
plt.pcolormesh(ordered, cmap="coolwarm")
plt.colorbar(label="Normalized similarity")
plt.title("Weisfeiler-Lehman subtree kernel (clustered)")
plt.gca().set_xticks(np.arange(ordered.shape[1]) + 0.5)
plt.gca().set_yticks(np.arange(ordered.shape[0]) + 0.5)
plt.gca().set_xticklabels(ordered.columns, rotation=90, fontsize=8)
plt.gca().set_yticklabels(ordered.index, fontsize=8)
plt.tight_layout()
plt.savefig("figures/plot.png")  # Save the figure

In [ ]:
import seaborn as sns

sns.clustermap(
    df_r,
    method="average",
    metric="correlation",
    cmap="coolwarm",
    figsize=(12, 12),
    yticklabels=True,
    xticklabels=True,
)
plt.title("Clustered Heatmap with Dendrograms")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
import numpy as np

# Set up the matplotlib figure
plt.figure(figsize=(15, 5))

# Create linkage matrices
row_linkage = linkage(df_r.values, method="average", metric="correlation")
col_linkage = linkage(df_r.values.T, method="average", metric="correlation")

# Plot row dendrogram
plt.subplot(1, 2, 1)
dendrogram(row_linkage, labels=df_r.index, orientation="left")
plt.title("Row Clustering")
plt.xlabel("Distance")

# Plot column dendrogram
# plt.subplot(1, 2, 2)
# dendrogram(col_linkage, labels=df_gk.columns)
# plt.title('Column Clustering')
# plt.xlabel('Distance')
# plt.xticks(rotation=90)

plt.savefig("figures/plot.svg")
plt.tight_layout()
plt.close()

In [ ]:
from scipy.cluster.hierarchy import fcluster


def analyze_clusters(linkage_matrix, labels, max_d=None, n_clusters=None):
    """
    Analyze clusters and their labels.
    Parameters:
        linkage_matrix: scipy linkage matrix
        labels: array of labels corresponding to the clustered items
        max_d: maximum distance for clustering (alternative to n_clusters)
        n_clusters: desired number of clusters (alternative to max_d)
    Returns:
        DataFrame with cluster analysis
    """
    # Get cluster assignments
    if max_d is not None:
        clusters = fcluster(linkage_matrix, max_d, criterion="distance")
    else:
        clusters = fcluster(linkage_matrix, n_clusters, criterion="maxclust")

    # Create DataFrame with labels and their clusters
    df_clusters = DataFrame({"label": labels, "cluster": clusters})

    # Group by cluster and aggregate labels
    cluster_summary = (
        df_clusters.groupby("cluster")
        .agg(
            size=("label", "count"),
            labels=("label", lambda x: list(x)),
        )
        .sort_values("size", ascending=False)
    )

    return cluster_summary


# Example usage for row clustering
n_clusters = 4  # adjust based on dendrogram inspection
cluster_summary = analyze_clusters(row_linkage, df_r.index, n_clusters=n_clusters)

# Print cluster analysis
print("\nCluster Analysis:")
for idx, row in cluster_summary.iterrows():
    print(f"\nCluster {idx} (size: {row['size']}):")
    # Print first few labels in each cluster
    print("Sample labels:", row["labels"][:5])

# Optional: Create a more compact visualization
plt.figure(figsize=(10, 5))
dendrogram(
    row_linkage,
    labels=df_r.index,
    orientation="left",
    leaf_rotation=0,
    leaf_font_size=8,
    truncate_mode="lastp",  # show only last p merged clusters
    p=30,  # show this many merges
    show_contracted=True,  # show collapsed sub-clusters
)
plt.title("Simplified Dendrogram with Major Clusters")
plt.tight_layout()
plt.show()

In [ ]:
import os
import json
import networkx as nx


def load_graphml_with_json_attrs(path: str) -> nx.Graph:
    """Read a GraphML file and attempt to JSON-decode any string attributes back into Python objects.

    Only replaces attribute values when json.loads returns a dict or list (to avoid converting plain strings).
    Works for Graph/DiGraph and MultiGraph/MultiDiGraph edge representations.
    """
    G = nx.read_graphml(path)

    # Nodes
    for n, d in G.nodes(data=True):
        for k, v in list(d.items()):
            if isinstance(v, str):
                try:
                    parsed = json.loads(v)
                    if isinstance(parsed, (dict, list)):
                        d[k] = parsed
                except Exception:
                    # leave as string if it isn't JSON
                    pass

    # Edges (handle keyed MultiGraphs and non-keyed graphs)
    try:
        edges = list(G.edges(keys=True, data=True))
        keyed = True
    except TypeError:
        edges = list(G.edges(data=True))
        keyed = False

    if keyed:
        for u, v, key, ed in edges:
            for k, val in list(ed.items()):
                if isinstance(val, str):
                    try:
                        parsed = json.loads(val)
                        if isinstance(parsed, (dict, list)):
                            ed[k] = parsed
                    except Exception:
                        pass
    else:
        for u, v, ed in edges:
            for k, val in list(ed.items()):
                if isinstance(val, str):
                    try:
                        parsed = json.loads(val)
                        if isinstance(parsed, (dict, list)):
                            ed[k] = parsed
                    except Exception:
                        pass

    return G


# Example usage: load the graphml written earlier and parse JSON attributes back
gml_path = path.replace(".json", ".graphml") if "path" in globals() else "ocel.graphml"
if os.path.exists(gml_path):
    ocel_nx_loaded = load_graphml_with_json_attrs(gml_path)
    print(
        f"Loaded GraphML from {gml_path} — nodes={ocel_nx_loaded.number_of_nodes()} edges={ocel_nx_loaded.number_of_edges()}"
    )
else:
    print(f"GraphML file not found: {gml_path}")
